# Notebook 04: Computer Vision - Image Classification

**Objective:** Learn how to classify images into categories using the Vision Transformer (ViT) architecture from HuggingFace.

**Prerequisites:** Python basics. No prior computer vision experience needed.

**Learning Objectives:**
- Understand how Vision Transformers (ViT) apply the transformer architecture to images
- Use the Pipeline API and direct model loading for image classification
- Classify images from URLs, local files, and standard datasets (CIFAR-10)
- Compare batch vs single-image classification performance
- Identify failure modes in pretrained image classifiers

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **CPU (Small)** | google/vit-base-patch16-224 | 346MB | 4GB | 4GB RAM, CPU | Good accuracy |
| **GPU (Medium)** | google/vit-large-patch16-224 | 1.2GB | 6GB | 8GB VRAM (RTX 4080) | Better accuracy |

### Software Requirements
- Python 3.10+
- Libraries: `transformers`, `torch`, `Pillow`, `pandas`, `torchvision`
- See `requirements.txt` for full list

## Overview

**Image classification** assigns a label from a predefined set of categories to an input image. It's the foundational task in computer vision -- object detection, segmentation, and captioning all build on classification.

**How Vision Transformers work:**

The key idea of ViT is to treat an image the same way BERT treats text:
1. Split the image into a grid of fixed-size **patches** (e.g., 16x16 pixels)
2. Flatten each patch into a vector and project it through a linear layer (like token embeddings)
3. Add **positional embeddings** so the model knows where each patch is
4. Prepend a special `[CLS]` token (just like BERT)
5. Process all patch tokens through a standard transformer encoder
6. Use the `[CLS]` token's output for classification

For a 224x224 image with 16x16 patches, the model processes $(224/16)^2 = 196$ patch tokens -- analogous to a 196-word sentence in NLP.

**ImageNet:** The model is pretrained on ImageNet, a dataset of 14M+ images across **1000 classes** including animals, vehicles, objects, food, and more.

## Expected Behaviors

### First Time Running
- **Model download**: ~346MB for ViT-base (2-4 minutes)
- Downloads model weights and image processor
- Cached in `~/.cache/huggingface/hub/`

### Classification Output
```python
[{'label': 'Egyptian cat', 'score': 0.8932},
 {'label': 'tabby cat', 'score': 0.0854}]
```
- Model outputs 1000 ImageNet classes, `top_k` controls how many you see

### Expected Accuracy
- **Clear, centered objects**: 80-95% confidence
- **Multiple objects**: Focuses on the most prominent one
- **Unusual angles/lighting**: 60-80% confidence
- **Objects not in ImageNet**: May misclassify

### Performance
- **Single image**: CPU 200-500ms, GPU 20-50ms
- **Batch of 10**: CPU 1-2s, GPU 100-200ms

## Setup and Installation

All imports, seeds, and environment checks. This is the first notebook that uses `Pillow` for image handling and `requests` for downloading images from URLs.

In [ ]:
import sys
import os
import time
import random
import warnings
from io import BytesIO

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
import requests
from PIL import Image
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    pipeline,
    set_seed,
)

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Selection

**ViT-base** splits images into 16x16 patches and processes them through a 12-layer transformer encoder with 86M parameters. It achieves 81.8% top-1 accuracy on ImageNet. The larger variant (ViT-large) has 24 layers and 304M parameters, reaching 84.5% accuracy at the cost of ~3x slower inference.

In [ ]:
# CHOOSE YOUR MODEL:

# Option 1: CPU-friendly (recommended for beginners)
MODEL_NAME = "google/vit-base-patch16-224"  # 346MB, ViT base

# Option 2: GPU-optimized (uncomment if you have RTX 4080 or similar)
# MODEL_NAME = "google/vit-large-patch16-224"  # 1.2GB, better accuracy

print(f"Selected model: {MODEL_NAME}")

### Helper Function: Image Loading

We need a utility to download images from URLs. This is used throughout the notebook for loading test images.

In [ ]:
def load_image_from_url(url: str, timeout: int = 10) -> Image.Image:
    """Download and open an image from a URL.

    Args:
        url: URL of the image to download.
        timeout: Request timeout in seconds.

    Returns:
        PIL Image object in RGB mode.

    Raises:
        requests.RequestException: If the download fails.
    """
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content))
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img

---

## Method 1: Using Pipeline (Simplest)

The `image-classification` pipeline works the same way as the NLP pipelines from Notebooks 01-03. Pass an image in (URL, file path, or PIL Image object), get predictions out. The pipeline handles resizing, normalization, and inference automatically.

In [ ]:
try:
    print(f"Loading {MODEL_NAME}...")
    classifier = pipeline(
        "image-classification",
        model=MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1,
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Check your internet connection for the first download.")
    raise

### Basic Image Classification

Let's classify a sample image. The `top_k` parameter controls how many predictions to return (sorted by confidence).

In [ ]:
def classify_image(
    image: Image.Image | str,
    top_k: int = 5,
) -> pd.DataFrame:
    """Classify an image and return top-k predictions as a DataFrame.

    Args:
        image: PIL Image, URL string, or file path.
        top_k: Number of top predictions to return.

    Returns:
        DataFrame with rank, label, and confidence score.
    """
    results = classifier(image, top_k=top_k)
    rows = [
        {"Rank": i, "Label": r["label"], "Confidence": round(r["score"], 4)}
        for i, r in enumerate(results, 1)
    ]
    return pd.DataFrame(rows)


# Load and classify a sample image
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"

try:
    image = load_image_from_url(image_url)
    print(f"Image size: {image.size}, Mode: {image.mode}")
    display(image)

    print("\nTop 5 Predictions:")
    pred_df = classify_image(image)
    display(pred_df)
except requests.RequestException as e:
    print(f"Could not download image: {e}")
    print("Try with a local image instead.")

The model returns ImageNet class names, which can be very specific ("Egyptian cat" vs "tabby cat" vs "tiger cat"). The confidence scores sum to approximately 1.0 across all 1000 classes.

### Classifying Multiple Images

Let's classify several images and compare results side by side.

In [ ]:
def classify_multiple_images(
    image_urls: list[str],
    top_k: int = 3,
) -> pd.DataFrame:
    """Classify multiple images from URLs and compile results.

    Args:
        image_urls: List of image URLs to classify.
        top_k: Number of top predictions per image.

    Returns:
        DataFrame with image index, top label, confidence, and runner-up.
    """
    rows: list[dict] = []
    for i, url in enumerate(image_urls, 1):
        try:
            img = load_image_from_url(url)
            results = classifier(img, top_k=top_k)
            rows.append({
                "Image": i,
                "Top Label": results[0]["label"],
                "Confidence": round(results[0]["score"], 4),
                "Runner-up": results[1]["label"] if len(results) > 1 else "-",
            })
        except Exception as e:
            rows.append({"Image": i, "Top Label": f"Error: {e}", "Confidence": 0.0, "Runner-up": "-"})

    return pd.DataFrame(rows)


test_urls = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg",
]

multi_df = classify_multiple_images(test_urls)
multi_df

---

## Method 2: Using Model and Processor Directly (Advanced)

Loading the model directly gives access to raw logits, attention weights, and the ability to batch process images efficiently. The **image processor** handles resizing (to 224x224), normalization, and conversion to tensors -- analogous to the tokenizer in NLP.

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(MODEL_NAME)
model = model.to(device)

print(f"Model loaded on: {device}")
print(f"Number of classes: {model.config.num_labels}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

With direct model access, we can get the full probability distribution across all 1000 classes and inspect the model's confidence in detail.

In [ ]:
def classify_with_full_probabilities(
    image: Image.Image,
    top_k: int = 5,
) -> pd.DataFrame:
    """Classify an image using direct model inference and return full probabilities.

    Args:
        image: PIL Image to classify.
        top_k: Number of top predictions to return.

    Returns:
        DataFrame with rank, label, and probability.
    """
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0]

    top_probs, top_indices = torch.topk(probs, k=top_k)

    rows = [
        {
            "Rank": i,
            "Label": model.config.id2label[idx.item()],
            "Probability": round(prob.item(), 6),
        }
        for i, (prob, idx) in enumerate(zip(top_probs, top_indices), 1)
    ]
    return pd.DataFrame(rows)


try:
    image = load_image_from_url(image_url)
    detail_df = classify_with_full_probabilities(image, top_k=10)
    print("Top 10 predictions with full probabilities:")
    display(detail_df)
except Exception as e:
    print(f"Could not process image: {e}")

Notice how the probability drops off sharply after the top prediction. The model is typically very confident about the primary class, with the remaining probability mass spread across similar classes.

---

## Practical Applications

### Example 1: Batch Classification

When you have many images to process, batch classification is much more efficient than processing one at a time. The processor pads images to the same size, and the model processes them in a single forward pass.

In [ ]:
def batch_classify(
    images: list[Image.Image],
) -> pd.DataFrame:
    """Classify a batch of images using direct model inference.

    Args:
        images: List of PIL Images to classify.

    Returns:
        DataFrame with image index and top predicted label.
    """
    inputs = processor(images=images, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        predictions = outputs.logits.argmax(dim=-1)

    rows = [
        {
            "Image": i,
            "Predicted Label": model.config.id2label[pred.item()],
        }
        for i, pred in enumerate(predictions, 1)
    ]
    return pd.DataFrame(rows)


batch_urls = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png",
]

try:
    batch_images = [load_image_from_url(url) for url in batch_urls]
    batch_result = batch_classify(batch_images)
    display(batch_result)
except Exception as e:
    print(f"Batch classification failed: {e}")

### Example 2: Confidence-Based Filtering

In production, you often want to only act on predictions the model is confident about and flag uncertain ones for human review.

In [ ]:
def classify_with_threshold(
    image: Image.Image,
    threshold: float = 0.5,
    top_k: int = 10,
) -> pd.DataFrame:
    """Return only predictions above a confidence threshold.

    Args:
        image: PIL Image to classify.
        threshold: Minimum confidence score to include.
        top_k: Number of top predictions to consider.

    Returns:
        DataFrame of predictions above the threshold.
    """
    results = classifier(image, top_k=top_k)
    filtered = [r for r in results if r["score"] >= threshold]

    if not filtered:
        filtered = [results[0]]  # Always include the top prediction

    return pd.DataFrame([
        {"Label": r["label"], "Confidence": round(r["score"], 4), "Above Threshold": r["score"] >= threshold}
        for r in filtered
    ])


try:
    image = load_image_from_url(image_url)
    print("Predictions above 30% confidence:")
    threshold_df = classify_with_threshold(image, threshold=0.3)
    display(threshold_df)
except Exception as e:
    print(f"Error: {e}")

### Example 3: Local Images

If you have images in the `sample_data/` directory, the classifier works on local files too.

In [ ]:
sample_data_path = "../sample_data"

if os.path.exists(sample_data_path):
    image_files = [
        f for f in os.listdir(sample_data_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    if image_files:
        rows: list[dict] = []
        for img_file in image_files[:5]:
            img = Image.open(os.path.join(sample_data_path, img_file)).convert("RGB")
            results = classifier(img, top_k=3)
            rows.append({
                "File": img_file,
                "Top Label": results[0]["label"],
                "Confidence": round(results[0]["score"], 4),
            })
        display(pd.DataFrame(rows))
    else:
        print("No images found in sample_data/. Add .jpg or .png files to test!")
else:
    print("sample_data/ directory not found. Skipping local image test.")

---

## Dataset Evaluation: CIFAR-10

Let's test the model on **CIFAR-10**, a standard benchmark with 60,000 32x32 color images across 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck).

There's an important mismatch here: CIFAR-10 images are tiny (32x32 pixels) while ViT expects 224x224. The processor upscales them, but this introduces blur. Also, ImageNet has 1000 classes while CIFAR-10 has only 10 -- the model may predict ImageNet-specific labels (e.g., "tabby cat" instead of "cat").

In [ ]:
def evaluate_on_cifar10(
    num_samples: int = 20,
) -> pd.DataFrame:
    """Evaluate the classifier on CIFAR-10 test images.

    Since ImageNet labels don't match CIFAR-10 labels exactly,
    we show both for manual comparison rather than computing accuracy.

    Args:
        num_samples: Number of test images to evaluate.

    Returns:
        DataFrame with true CIFAR-10 class, predicted ImageNet label, and confidence.
    """
    import torchvision.datasets as datasets

    cifar_classes = [
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck",
    ]

    print("Downloading CIFAR-10 test set...")
    cifar10_test = datasets.CIFAR10(root="./data", train=False, download=True)
    print(f"Loaded {len(cifar10_test)} test images\n")

    rows: list[dict] = []
    for i in range(num_samples):
        img, true_label = cifar10_test[i]
        results = classifier(img, top_k=1)
        rows.append({
            "Image": i + 1,
            "True Class (CIFAR-10)": cifar_classes[true_label],
            "Predicted (ImageNet)": results[0]["label"],
            "Confidence": round(results[0]["score"], 4),
        })

    return pd.DataFrame(rows)


try:
    cifar_df = evaluate_on_cifar10(num_samples=10)
    display(cifar_df)
except Exception as e:
    print(f"Could not load CIFAR-10: {e}")
    print("Install with: pip install torchvision")

The label mismatch is expected. ImageNet's "airliner" maps to CIFAR-10's "airplane", "tabby cat" to "cat", etc. Despite the low resolution (32x32 upscaled to 224x224), the model often gets the correct category.

---

## Performance Benchmarking

Let's measure how classification speed scales with the number of images processed at once.

In [ ]:
def benchmark_classification(
    batch_sizes: list[int] | None = None,
    num_runs: int = 3,
) -> pd.DataFrame:
    """Benchmark image classification speed across batch sizes.

    Uses a synthetic image to avoid download overhead.

    Args:
        batch_sizes: List of batch sizes to test. Defaults to [1, 5, 10].
        num_runs: Number of runs to average per batch size.

    Returns:
        DataFrame with batch size, avg time, and images per second.
    """
    if batch_sizes is None:
        batch_sizes = [1, 5, 10]

    # Create a synthetic test image
    test_image = Image.new("RGB", (224, 224), color=(128, 128, 128))

    rows: list[dict] = []
    for batch_size in batch_sizes:
        images = [test_image] * batch_size
        times: list[float] = []

        for _ in range(num_runs):
            start = time.time()
            classifier(images, top_k=1)
            times.append(time.time() - start)

        avg_time = np.mean(times)
        rows.append({
            "Batch Size": batch_size,
            "Avg Time (s)": round(avg_time, 3),
            "Images/sec": round(batch_size / avg_time, 1),
            "Device": "GPU" if torch.cuda.is_available() else "CPU",
        })

    return pd.DataFrame(rows)


print(f"Benchmarking {MODEL_NAME} on {device}...\n")
bench_df = benchmark_classification()
bench_df

Throughput (images/sec) increases with batch size because the model amortizes its fixed overhead. On GPU, the improvement is more dramatic since GPU parallelism is better utilized with larger batches.

---

## Error Analysis: Where Image Classifiers Fail

Pretrained image classifiers have well-known failure modes. Understanding these helps you decide when to trust the model and when to add safeguards.

In [ ]:
def analyze_failure_modes() -> pd.DataFrame:
    """Test the classifier on synthetic images that expose common failure modes.

    Returns:
        DataFrame showing failure category, input description, and model prediction.
    """
    test_cases: list[tuple[Image.Image, str, str]] = []

    # Solid color -- no object present
    test_cases.append((Image.new("RGB", (224, 224), (255, 0, 0)), "Solid red image", "No object"))
    test_cases.append((Image.new("RGB", (224, 224), (0, 0, 0)), "Solid black image", "No object"))

    # Random noise -- adversarial-like input
    noise = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    test_cases.append((Image.fromarray(noise), "Random noise", "Noise"))

    # Very small object in large blank field
    tiny_obj = Image.new("RGB", (224, 224), (255, 255, 255))
    # Draw a small red square in the corner
    for x in range(5):
        for y in range(5):
            tiny_obj.putpixel((x, y), (255, 0, 0))
    test_cases.append((tiny_obj, "Tiny red dot on white", "Tiny object"))

    rows: list[dict] = []
    for img, description, category in test_cases:
        results = classifier(img, top_k=1)
        rows.append({
            "Category": category,
            "Input": description,
            "Predicted Label": results[0]["label"],
            "Confidence": round(results[0]["score"], 4),
        })

    return pd.DataFrame(rows)


print("Failure Mode Analysis:\n")
failure_df = analyze_failure_modes()
failure_df

**What to notice:**

- **No object present**: The model *always* predicts something from its 1000 classes, even for blank images. It has no concept of "none of the above" or "I don't know." In production, use confidence thresholds to reject low-confidence predictions.
- **Random noise**: The model may assign surprisingly high confidence to random noise. This is a known vulnerability of neural networks.
- **Tiny objects**: The model focuses on global image statistics. Very small objects in large backgrounds may be ignored entirely.

**Additional known limitations:**
- **ImageNet bias**: The model only knows its 1000 training classes. An image of a "smartphone" will be classified as "cellular telephone" or similar -- close but not always intuitive.
- **Cultural bias**: ImageNet is skewed toward Western objects and contexts.
- **Adversarial examples**: Small, imperceptible pixel changes can flip predictions with high confidence.

---

## Exercises

1. **Your own images**: Classify photos from your phone or camera using the local image example. How accurate is the model on your images?

2. **Ambiguous images**: Find images that could reasonably belong to multiple ImageNet classes (e.g., a wolf vs a husky). What does the top-5 distribution look like?

3. **Model comparison**: If you have GPU, compare `vit-base` with `vit-large` on the same images. Is the accuracy difference noticeable?

4. **ResNet comparison**: Try `microsoft/resnet-50` instead of ViT. How do the predictions differ?

5. **Batch scaling**: Use `benchmark_classification()` with batch sizes `[1, 2, 4, 8, 16, 32]`. At what batch size does throughput plateau on your hardware?

---

## State-of-the-Art Open Models (Reference)

| Model | Developer | Size | ImageNet Accuracy | Best For |
|-------|-----------|------|-------------------|----------|
| **ViT-Base** | Google | 346MB | 81.8% | Learning, general use |
| **DeiT-Base** | Meta | 346MB | 85.2% | Data-efficient training |
| **EfficientNetV2** | Google | 264MB | 87.3% | Production deployment |
| **ConvNeXt-Large** | Meta | 800MB | 87.8% | High accuracy, pure CNN |
| **Swin-Large** | Microsoft | 800MB | 87.3% | High-res images, detection |
| **BEiT-Large** | Microsoft | 1.2GB | 88.6% | Transfer learning |

For production, **ConvNeXt** and **Swin** offer the best accuracy-efficiency trade-off with GPU acceleration. **EfficientNetV2** is fastest for CPU deployment.

---

## Key Takeaways

- **Vision Transformer (ViT)** treats images as sequences of patches, applying the same transformer architecture used in NLP
- **ImageNet pretraining** enables recognition of 1000 object categories out of the box
- **Batch processing** is significantly faster than single-image classification, especially on GPU
- The model **always predicts a class** even for invalid inputs -- use confidence thresholds to filter uncertain predictions
- **Image resolution matters**: low-res inputs (like CIFAR-10's 32x32) are upscaled but lose detail

## Next Steps

- **Notebook 05 (CV): Object Detection** -- Locate and label multiple objects within a single image
- **Notebook 06 (CV): OCR** -- Extract text from images
- [HuggingFace Image Classification Models](https://huggingface.co/models?pipeline_tag=image-classification)

## Resources

- [Vision Transformer Paper](https://arxiv.org/abs/2010.11929)
- [Image Classification Guide](https://huggingface.co/docs/transformers/tasks/image_classification)
- [ImageNet Classes](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a)